In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns

In [2]:
import os 
print(os.getcwd())

d:\BookRecommendation\notebook


In [4]:
import pandas as pd

ratings_df    = pd.read_csv(r'd:\BookRecommendation\data\processed\clean_book_ratings.csv')
user_features = pd.read_csv(r'd:\BookRecommendation\data\processed\user_features.csv')
book_features = pd.read_csv(r'd:\BookRecommendation\data\processed\book_features.csv')
user_clusters = pd.read_csv(r'd:\BookRecommendation\data\processed\user_clusters.csv')
book_clusters = pd.read_csv(r'd:\BookRecommendation\data\processed\book_clusters.csv')
cf_scores_df  = pd.read_csv(r'd:\BookRecommendation\data\processed\cf_scores.csv')

print("Ratings columns:      ", ratings_df.columns.tolist())
print("User features columns:", user_features.columns.tolist())
print("Book features columns:", book_features.columns.tolist())
print("User clusters columns:", user_clusters.columns.tolist())
print("Book clusters columns:", book_clusters.columns.tolist())
print("CF scores columns:    ", cf_scores_df.columns.tolist())

Ratings columns:       ['index', 'ISBN', 'Book_Title', 'Book_Author', 'Year_Of_Publication', 'Publisher', 'User_ID', 'Book_Rating', 'Age_Group', 'Country']
User features columns: ['User_ID', 'user_avg_rating', 'user_rating_count', 'user_rating_std', 'Age_Group', 'Country', 'favourite_author', 'old_books_ratio', 'recent_books_ratio', 'genre_diversity']
Book features columns: ['ISBN', 'Book_avg_rating', 'Book_rating_count', 'Book_rating_std', 'Year_Of_Publication', 'Publisher', 'Genre', 'Title_Author_combined', 'Genre_grouped']
User clusters columns: ['User_ID', 'user_avg_rating', 'user_rating_count', 'user_rating_std', 'Age_Group', 'Country', 'favourite_author', 'old_books_ratio', 'recent_books_ratio', 'genre_diversity', 'user_cluster']
Book clusters columns: ['ISBN', 'Book_avg_rating', 'Book_rating_count', 'Book_rating_std', 'Year_Of_Publication', 'Publisher', 'Genre', 'Title_Author_combined', 'Genre_grouped', 'book_cluster']
CF scores columns:     ['User_ID', 'ISBN', 'cf_predicted_sco

In [5]:
xgb_df = ratings_df[ratings_df['Book_Rating']>0][
    ['User_ID','ISBN','Book_Author','Book_Rating']].copy().reset_index(drop=True)


In [7]:
user_cols = ['User_ID', 'user_avg_rating', 'user_rating_count', 'user_rating_std', 'Age_Group', 'Country', 'favourite_author', 'old_books_ratio', 'recent_books_ratio', 'genre_diversity']

xgb_df = xgb_df.merge(user_features[user_cols], on='User_ID', how='left')

In [25]:
xgb_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 63345 entries, 0 to 63344
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   User_ID               63345 non-null  int64  
 1   ISBN                  63345 non-null  str    
 2   Book_Author           63345 non-null  str    
 3   Book_Rating           63345 non-null  int64  
 4   user_avg_rating       63345 non-null  float64
 5   user_rating_count     63345 non-null  int64  
 6   user_rating_std       63345 non-null  float64
 7   Age_Group             63345 non-null  str    
 8   Country               63345 non-null  str    
 9   favourite_author      63345 non-null  str    
 10  old_books_ratio       63345 non-null  float64
 11  recent_books_ratio    63345 non-null  float64
 12  genre_diversity       63345 non-null  int64  
 13  user_cluster          63345 non-null  int64  
 14  Book_avg_rating       63345 non-null  float64
 15  Book_rating_count     63345 no

In [10]:
xgb_df = xgb_df.merge(user_clusters[['User_ID','user_cluster']]
                      ,on='User_ID',
                      how='left')

In [12]:
book_cols = ['ISBN', 'Book_avg_rating', 'Book_rating_count',
             'Book_rating_std', 'Year_Of_Publication', 'Genre_grouped']


xgb_df = xgb_df.merge(book_features[book_cols] , on='ISBN' , how='left')

In [13]:
xgb_df = xgb_df.merge(book_clusters[['ISBN','book_cluster']] , on='ISBN' , how='left')


In [20]:
xgb_df = xgb_df.merge(cf_scores_df , on=['User_ID','ISBN'] , how='left')

In [17]:
print("xgb_df types:")
print(xgb_df[['User_ID', 'ISBN']].dtypes)

print("\ncf_scores_df types:")
print(cf_scores_df[['User_ID', 'ISBN']].dtypes)

xgb_df types:
User_ID    int64
ISBN         str
dtype: object

cf_scores_df types:
User_ID    int64
ISBN         str
dtype: object


In [18]:
!pip install tqdm

  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
Using cached tqdm-4.67.3-py3-none-any.whl (78 kB)


In [19]:
import joblib
from tqdm import tqdm
tqdm.pandas()

# Load SVD model if not already loaded
svd_model = joblib.load(r'd:\BookRecommendation\ml_models\cf_svd_model.pkl')

# Predict CF score for each explicit rating pair
xgb_df['cf_predicted_score'] = xgb_df.progress_apply(
    lambda row: svd_model.predict(
        uid=row['User_ID'],
        iid=row['ISBN']
    ).est, axis=1
)

print(xgb_df['cf_predicted_score'].describe())

100%|██████████| 63345/63345 [00:00<00:00, 101246.21it/s]

count    63345.000000
mean         7.842065
std          1.017684
min          1.381996
25%          7.281717
50%          7.915875
75%          8.526259
max         10.000000
Name: cf_predicted_score, dtype: float64


In [24]:
xgb_df.columns

Index(['User_ID', 'ISBN', 'Book_Author', 'Book_Rating', 'user_avg_rating',
       'user_rating_count', 'user_rating_std', 'Age_Group', 'Country',
       'favourite_author', 'old_books_ratio', 'recent_books_ratio',
       'genre_diversity', 'user_cluster', 'Book_avg_rating',
       'Book_rating_count', 'Book_rating_std', 'Year_Of_Publication',
       'Genre_grouped', 'book_cluster', 'cf_predicted_score_x'],
      dtype='str')

In [23]:
xgb_df.drop(columns=['cf_predicted_score_y'] , inplace=True)

In [26]:
# Genre match — does user's favourite author match book author?
xgb_df['author_match'] = (
    xgb_df['favourite_author'] == xgb_df['Book_Author']
).astype(int)

# Popularity signal
median_count = book_features['Book_rating_count'].median()
xgb_df['is_popular'] = (
    xgb_df['Book_rating_count'] > median_count
).astype(int)

# Implicit interaction signal
implicit = ratings_df[ratings_df['Book_Rating'] == 0][
    ['User_ID', 'ISBN']
].copy()
implicit['interacted'] = 1

xgb_df = xgb_df.merge(implicit, on=['User_ID', 'ISBN'], how='left')
xgb_df['interacted'] = xgb_df['interacted'].fillna(0).astype(int)

print(f"After interaction features: {xgb_df.shape}")

After interaction features: (63345, 24)


In [33]:
xgb_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 63345 entries, 0 to 63344
Data columns (total 20 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   User_ID              63345 non-null  int64  
 1   Book_Rating          63345 non-null  int64  
 2   user_avg_rating      63345 non-null  float64
 3   user_rating_count    63345 non-null  int64  
 4   user_rating_std      63345 non-null  float64
 5   Age_Group            63345 non-null  str    
 6   old_books_ratio      63345 non-null  float64
 7   recent_books_ratio   63345 non-null  float64
 8   genre_diversity      63345 non-null  int64  
 9   user_cluster         63345 non-null  int64  
 10  Book_avg_rating      63345 non-null  float64
 11  Book_rating_count    63345 non-null  int64  
 12  Book_rating_std      63345 non-null  float64
 13  Year_Of_Publication  63345 non-null  int64  
 14  Genre_grouped        63345 non-null  str    
 15  book_cluster         63345 non-null  int64  
 1

In [30]:
xgb_df.rename(columns={'cf_predicted_score_x':'cf_predicted_score'} ,inplace=True)

In [32]:
xgb_df.drop(columns=['ISBN', 'Book_Author', 
                      'favourite_author', 'Country'], inplace=True)